# OpenPlaque — Supervised Boundary Parameter Tuning with Cached Predictions

Clean Colab workflow using OpenPlaque code directly from GitHub.

This notebook **does not use cross-validation**. It uses all labeled sample cases to select deterministic post-processing parameters.

Key optimization: nnU-Net prediction is run **once per sample case**. The predictions are cached locally and also saved as a compressed ZIP archive in Google Drive. On a future clean Colab run, the notebook restores predictions from that archive and skips nnU-Net inference.

Expected labeled sample-data pattern:

```text
Sample_Dataset/
  P02_LAD_axial_0000.nii.gz   # CT input image
  P02_LAD_axial.nii.gz        # expert label mask, values 0/1/2
```

or nnU-Net raw layout:

```text
Dataset001_CCTA_DHM/
  imagesTr/*_0000.nii.gz
  labelsTr/*.nii.gz
```

Outputs:

```text
/content/drive/MyDrive/OpenPlaque/Boundary_Parameter_Tuning/
  predictions/
  nnunet_prediction_cache.zip
  parameter_summary.csv
  best_boundary_parameters.json
```

Research use only. Not for clinical decision-making.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone/update OpenPlaque from GitHub.
# This notebook assumes the repository already contains:
#   src/openplaque/boundary.py
#   src/openplaque/boundary_parameter_tuning.py

import sys
from pathlib import Path

repo = Path('/content/OpenPlaque')
if not repo.exists():
    !git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
else:
    !git -C /content/OpenPlaque pull

src = repo / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print('Using OpenPlaque source:', src)

In [ ]:
# Install dependencies.
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
!pip install -q pandas scipy matplotlib SimpleITK pydicom gdown

# Verify GPU for nnU-Net prediction cache generation.
!nvidia-smi

In [ ]:
# Configure nnU-Net paths.
import os
from pathlib import Path

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'

for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('nnUNet_results:', os.environ['nnUNet_results'])

In [ ]:
# Import and verify the repository additions.
import openplaque
from openplaque.boundary_parameter_tuning import (
    DEFAULT_GRID,
    SMALL_GRID,
    collect_sample_cases,
    detect_available_model_folds,
    ensure_prediction_cache_with_archive,
    prediction_cache_status,
    parameter_grid_dataframe,
    parameter_grid_size,
    evaluate_all_cases,
    aggregate_by_params,
    select_best_params,
    save_parameter_evaluation_outputs,
)

print('openplaque:', openplaque.__file__)
print('Boundary parameter tuning module loaded')

## 1. Load or download the labeled sample dataset

The old tuning notebook used a Google Drive folder named `Sample_Dataset`. This cell first checks Drive/local paths, then downloads the old sample folder if needed.

In [ ]:
from pathlib import Path

DATASET_CANDIDATES = [
    Path('/content/drive/MyDrive/OpenPlaque/Sample_Dataset'),
    Path('/content/drive/MyDrive/OpenPlaque/Dataset001_CCTA_DHM'),
    Path('/content/sample_dataset/Sample_Dataset'),
    Path('/content/Dataset001_CCTA_DHM'),
]

DATASET_DIR = next((p for p in DATASET_CANDIDATES if p.exists()), None)

if DATASET_DIR is None:
    print('No local sample dataset found. Downloading old OpenPlaque sample dataset from Google Drive...')
    !rm -rf /content/sample_dataset
    !gdown --folder "https://drive.google.com/drive/folders/1i4XD-GfP-wS50m9smGR1LzBRJokKro9_?usp=sharing" -O /content/sample_dataset --remaining-ok
    DATASET_DIR = Path('/content/sample_dataset/Sample_Dataset')

if DATASET_DIR is None or not DATASET_DIR.exists():
    raise FileNotFoundError('Could not find or download the labeled sample dataset.')

cases = collect_sample_cases(DATASET_DIR)
print('Using dataset:', DATASET_DIR)
print('Paired image/label cases:', len(cases))
for case in cases[:10]:
    print(case.case_id, 'image=', case.image_path.name, 'label=', case.label_path.name)

## 2. Show the exact parameter grid

The default grid includes the original parameters plus added morphology/connectivity parameters.

In [ ]:
# Choose grid size.
# USE_SMALL_GRID=True is useful for a quick smoke test.
# Set USE_SMALL_GRID=False for the full grid.
USE_SMALL_GRID = False
parameter_grid = SMALL_GRID if USE_SMALL_GRID else DEFAULT_GRID

grid_df = parameter_grid_dataframe(parameter_grid)
print('Parameter grid size:', parameter_grid_size(parameter_grid))
print('Grid values:')
for name, values in parameter_grid.items():
    print(f'  {name}: {values}')

display(grid_df.head(30))

## 3. Extract trained nnU-Net model weights

In [ ]:
# Extract model weights if needed.
import zipfile

model_zip = Path('/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip')
model_target = Path('/content/nnUNet_results/Dataset001_CCTA_DHM')

if model_target.exists():
    print('Model already extracted:', model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f'Missing model ZIP: {model_zip}')
    print('Extracting model ZIP...')
    with zipfile.ZipFile(model_zip) as z:
        z.extractall('/content/nnUNet_results')
    print('Extracted model to /content/nnUNet_results')

folds = detect_available_model_folds('/content/nnUNet_results', dataset_name='Dataset001_CCTA_DHM')
print('Available nnU-Net folds:', folds)
!find /content/nnUNet_results -maxdepth 4 | head -80

## 4. Generate, restore, or reuse cached predictions

This is the key runtime optimization. nnU-Net runs once per labeled case. The predictions are saved in two places:

1. Local/Drive folder: `predictions/`
2. Compressed Drive archive: `nnunet_prediction_cache.zip`

On a future clean Colab run, the notebook restores `predictions/` from the compressed archive and skips nnU-Net inference.

In [ ]:
output_dir = Path('/content/drive/MyDrive/OpenPlaque/Boundary_Parameter_Tuning')
pred_dir = output_dir / 'predictions'
input_dir = output_dir / 'prediction_inputs'
prediction_archive = output_dir / 'nnunet_prediction_cache.zip'
output_dir.mkdir(parents=True, exist_ok=True)

print('Prediction archive:', prediction_archive)
print('Archive exists:', prediction_archive.exists())
print('Prediction cache before inference/restore:')
display(prediction_cache_status(cases, pred_dir).head(20))

# Set to True only if you intentionally want to rerun nnU-Net and refresh the archive.
OVERWRITE_PREDICTIONS = False
OVERWRITE_ARCHIVE = False

ensure_prediction_cache_with_archive(
    cases=cases,
    pred_dir=pred_dir,
    input_dir=input_dir,
    archive_path=prediction_archive,
    dataset_name_or_id='Dataset001_CCTA_DHM',
    configuration='3d_fullres',
    folds=folds,
    overwrite_predictions=OVERWRITE_PREDICTIONS,
    overwrite_archive=OVERWRITE_ARCHIVE,
)

print('Prediction cache after inference/restore:')
display(prediction_cache_status(cases, pred_dir).head(20))
print('Archive exists after:', prediction_archive.exists())

## 5. Evaluate every parameter set on all labeled cases

In [ ]:
all_case_results = evaluate_all_cases(cases, pred_dir=pred_dir, parameter_grid=parameter_grid)
summary = aggregate_by_params(all_case_results)
final_params = select_best_params(all_case_results)

print('Evaluated rows:', len(all_case_results))
print('Summary parameter sets:', len(summary))
print('Best parameters selected on all labeled cases:')
for k, v in final_params.items():
    print(f'  {k}: {v}')

display(summary.head(20))

## 6. Save outputs

In [ ]:
paths = save_parameter_evaluation_outputs(
    all_case_results=all_case_results,
    final_params=final_params,
    output_dir=output_dir,
)

print('Saved outputs:')
for name, path in paths.items():
    print(f'  {name}: {path}')

## 7. Optional: inspect per-case results for the selected parameters

In [ ]:
# Null-safe filter to selected parameter rows.
selected_mask = True
for col, val in final_params.items():
    if col not in all_case_results.columns:
        continue
    if val is None:
        selected_mask = selected_mask & all_case_results[col].isna()
    else:
        selected_mask = selected_mask & (all_case_results[col] == val)

selected_case_results = all_case_results[selected_mask].sort_values('case_id')
display(selected_case_results[[
    'case_id', 'dice', 'iou', 'precision', 'recall',
    'abs_tpv_error_fraction', 'true_tpv_mm3', 'pred_tpv_mm3',
    'raw_pred_tpv_mm3', 'refined_tpv_mm3', 'removed_tpv_mm3', 'score'
]])